# **Sentiment Analysis of 3 Retail Securities Brokers from Google Play Store (Stockbit, Ajaib, Indo Premier)**
authour = Rafi King Akbar 

## **Scrapping and Merger Data**

In [ ]:
!pip install google_play_scraper

In [ ]:
import pandas as pd
from google_play_scraper import reviews, Sort

apps_id = {
    "Ajaib": "ajaib.co.id",
    "IPOT": "com.indopremier.ipot",
    "Stockbit": "com.stockbit.android"
}

N = 2000

#Algoritma Scrapping
def get_reviews(app_id, app_name, n_review = N):
  result,_= reviews(
      app_id,
      lang = 'id',
      country = 'id',
      sort = Sort.NEWEST,
      count = n_review
  )
  df = pd.DataFrame(result)

  #Pemilihan kolom yang penting
  kolom = [
        "reviewId", "userName", "score", "at",
        "content", "replyContent", "thumbsUpCount", "appVersion"
  ]
  df=df[kolom]
  df["app_name"]= app_name
  return df

#Scrap Keseluruhan Aplikasi
list_df = []

for nama_app, app_id in apps_id.items():
    print(f"Sedang mengambil review: {nama_app}")
    df_app = get_reviews(app_id, nama_app)
    print(f"-> didapat {len(df_app)} baris\n")
    list_df.append(df_app)

#Merge
df_all = pd.concat(list_df, ignore_index=True)

In [ ]:
#cek isi datanya per aplikasi
df_all[df_all["app_name"] == "Stockbit"]

In [ ]:
#cek isi datanya semua aplikasi/merger
print("Jumlah total baris:", len(df_all))
df_all

In [ ]:
#penyimpanan ke CSV
output_path = "reviews_all_sekuritas_raw.csv"
df_all.to_csv(output_path, index=False)

#**Import Data**


Scrapping dilakukan pada **18/11/2025**, dan tahap ini merupakan proses pengambilan data awal sebanyak **2000 review untuk setiap aplikasi**. Ditetapkan agar hasilnya tidak berubah-ubah

In [ ]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/rafikingakbar/SentimenAnalysisRetailSecurities/refs/heads/main/data/reviews_all_sekuritas_raw.csv")
df

#**Preprocessing**

In [ ]:
df.info()

###1.Lower Casing


In [ ]:
pre1 = []

for i in df['content']:
  hasil_lc = str.lower(i)
  pre1.append(hasil_lc)

beforeafter = pd.DataFrame({'app_name':df['app_name'],'Sebelum Preprocessing': df['content'], 'Setelah Lower Casing ': pre1})
beforeafter

###2.Removal HTML tags

In [ ]:
import re
def remove_html(text):
  html_pattern = re.compile('<.*?>')
  return html_pattern.sub(r'', text)

pre2 = []
for i in beforeafter['Setelah Lower Casing ']:
  hasil_rht = remove_html(i)
  pre2.append(hasil_rht)

beforeafter ['Setelah Remove HTML Tags'] = pre2
beforeafter

###3.Remove Punctuation

In [ ]:
import string
punc = string.punctuation

def remove_punctuation(text):
  return text.translate(str.maketrans('','',punc))

pre3=[]
for i in beforeafter['Setelah Remove HTML Tags']:
  hasil_rp = remove_punctuation(i)
  pre3.append(hasil_rp)

beforeafter['Setelah Remove Punctuation'] = pre3
beforeafter

###4.Remove Emoji

In [ ]:
!pip install demoji

In [ ]:
import demoji

def remove_emoji(text):
  cleaned_text = demoji.replace(text,repl='')
  return cleaned_text

pre4 = []
for i in beforeafter['Setelah Remove Punctuation']:
  hasil_re = remove_emoji(i)
  pre4.append(hasil_re)

beforeafter["Setelah Remove Emoji"] = pre4
beforeafter


###5.Normalize

a. Tokenisasi Sederhana

In [ ]:
all_words = " ".join(beforeafter['Setelah Remove Emoji']).split()

b. Melihat dan Hitung Frekuensi Kata

In [ ]:
from collections import Counter

freq = Counter(all_words)
freq.most_common(200)

c. Algoritma Normalize

In [ ]:
norm = {
    "yg": "yang",
    "gk": "tidak",
    "ga": "tidak",
    "gak": "tidak",
    "nggak": "tidak",
    "bgt": "banget",
    "aja": "saja",
    "kalo": "kalau",
    "udah": "sudah",
    "udh": "sudah"
}

def normalisasi(text):
  words = text.split()
  hasil = []

  for w in words:
    if w in norm:
      hasil.append(norm[w])
    else:
      hasil.append(w)
  return " ".join(hasil)

pre5 = []
for i in beforeafter['Setelah Remove Emoji']:
  hasil_norm = normalisasi(i)
  pre5.append(hasil_norm)

beforeafter['Setelah Normalize'] = pre5
beforeafter


#**Klasifikasi Sentimen (Indo RoBERTa)**

###1.Simpan Data Preprocessing Terakhir

a. Masukan ke CSV

In [ ]:
df_norm = beforeafter[["app_name","Setelah Normalize"]]
df_norm

In [ ]:
df_norm.to_csv("reviews_data_normalized.csv", index=False)

b. Baca Data

In [ ]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/rafikingakbar/SentimenAnalysisRetailSecurities/refs/heads/main/data/reviews_data_normalized.csv")
df.head()

c. Ubah Nama Kolom

In [ ]:
df = df.rename(columns={"Setelah Normalize": "review_all"})
df.head()

d. Cek Baris Kosong

In [ ]:
df[df['review_all'].isna()]

e. Drop Baris Kosong

In [ ]:
df = df.dropna(subset=['review_all'])

In [ ]:
df['review_all'].apply(type).value_counts()


###2.Klasifikasi Sentimen

a. Algoritma Labelling

In [ ]:
from transformers import pipeline

classifier = pipeline('sentiment-analysis', model='w11wo/indonesian-roberta-base-sentiment-classifier')

def prediksi_sentimen(teks):
  hasil = classifier(teks)
  return hasil[0]['label']

df['sentimen'] = df['review_all'].apply(prediksi_sentimen)

b. Melihat Jumlah Label Sentimen

In [ ]:
sentimen_counts = df.sentimen.value_counts()
sentimen_counts

c. simpan dalam bentuk CSV

In [ ]:
df.to_csv("reviews_data_SentimenLabelling.csv", index=False)

#**Evaluasi Auto Labelling**

###1.Baca Data

In [ ]:
import pandas as pd

df_sl=pd.read_csv("https://raw.githubusercontent.com/rafikingakbar/SentimenAnalysisRetailSecurities/refs/heads/main/data/reviews_data_SentimenLabelling.csv")
df_sl.head()

###2.Ambil 300 Data Random

In [ ]:
df_eval = df_sl.sample(300, random_state=42).copy()
df_eval["label_manual"]=""
df_eval.to_csv("evaluasi_300_data.csv", index=False)
df_eval.head()


###3.Load File Yang Sudah Diisi Manual

In [ ]:
import pandas as pd

df_eval_done = pd.read_csv("https://raw.githubusercontent.com/rafikingakbar/SentimenAnalysisRetailSecurities/refs/heads/main/data/evaluasi_300_data.csv", sep=";")
df_eval_done

###4.Evaluasi Auto Labelling

a. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(df_eval_done["label_manual"], df_eval_done["sentimen"])

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=["negative","neutral","positive"],
            yticklabels=["negative","neutral","positive"],
            cmap="Blues")
plt.xlabel("Predicted (Auto-Label)")
plt.ylabel("Actual (Manual)")
plt.title("Confusion Matrix Evaluasi Labeling (300 Data)")
plt.show()

b. Ukuran Evaluasi

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(df_eval_done["label_manual"],
                                 df_eval_done["sentimen"]))

print("\nClassification Report:\n")
print(classification_report(df_eval_done["label_manual"],
                            df_eval_done["sentimen"]))


c. Melihat apa saja yang berbeda

In [ ]:
df_error = df_eval_done[df_eval_done["label_manual"] != df_eval_done["sentimen"]]
df_error


#**Analisis Sentimen Global**

###1.Baca Data

In [ ]:
import pandas as pd

df_sl=pd.read_csv("https://raw.githubusercontent.com/rafikingakbar/SentimenAnalysisRetailSecurities/refs/heads/main/data/reviews_data_SentimenLabelling.csv")
df_sl.head()

###2.Visualisasi Sentimen

a. Jumlah Sentimen Keseluruhan

In [ ]:
df_sl['sentimen'].value_counts()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(data=df_sl, x='sentimen')
plt.title("Distribusi Sentimen Secara Global")
plt.show()

b. Jumlah Sentimen Per Aplikasi

In [ ]:
sentimen_app = df_sl.groupby(['app_name', 'sentimen']).size().unstack(fill_value=0)
sentimen_app

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sentimen_app.plot(kind='bar', figsize=(8,5))
plt.title("Perbandingan Sentimen per Aplikasi")
plt.xlabel("Aplikasi")
plt.ylabel("Jumlah Review")
plt.show()

c. Piechart Per Aplikasi - Negatif

In [ ]:
import matplotlib.pyplot as plt

neg_count = df_sl[df_sl['sentimen'] == 'negative']['app_name'].value_counts()

plt.figure(figsize=(6,6))
plt.pie(neg_count, labels=neg_count.index, autopct='%1.1f%%')
plt.title("Proporsi Ulasan Negatif per Aplikasi")
plt.show()

d. Wordcloud tiap aplikasi

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

apps = df_sl['app_name'].unique()

for app in apps:
    text = " ".join(df_sl[df_sl['app_name']==app]['review_all'])
    wc = WordCloud(width=800, height=500, background_color="white").generate(text)
    plt.figure(figsize=(7,4))
    plt.imshow(wc)
    plt.axis("off")
    plt.title(f"Wordcloud - {app}")
    plt.show()

e. Wordcloud Sentimen per Aplikasi

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

df_pos = df_sl[df_sl["sentimen"] == "positive"]
df_neg = df_sl[df_sl["sentimen"] == "negative"]

apps = df_sl["app_name"].unique()

for app in apps:

    text_pos = " ".join(df_pos[df_pos["app_name"] == app]["review_all"].dropna())
    text_neg = " ".join(df_neg[df_neg["app_name"] == app]["review_all"].dropna())

    #Wordcloud Pos
    if text_pos.strip():
        wc_pos = WordCloud(
            width=800,
            height=400,
            background_color="white",
            colormap="Greens"
        ).generate(text_pos)

        plt.figure()
        plt.imshow(wc_pos)
        plt.axis("off")
        plt.title(f"WordCloud Positif - {app}")
        plt.show()

    #Worcloud Neg
    if text_neg.strip():
        wc_neg = WordCloud(
            width=800,
            height=400,
            background_color="white",
            colormap="Reds"
        ).generate(text_neg)

        plt.figure()
        plt.imshow(wc_neg)
        plt.axis("off")
        plt.title(f"WordCloud Negatif - {app}")
        plt.show()

#**PreProcessing Before Clustering**

###1.Retrieving Negative Data

In [ ]:
df_slneg = df_sl[df_sl["sentimen"] == "negative"]
df_slneg

In [ ]:
#Melihat jumlah komen negatif per aplikasi
app_negcount = df_slneg['app_name'].value_counts()
app_negcount

In [ ]:
df_slneg.to_csv("reviews_data_SentimenNegative.csv", index=False)

###2.Load data dari Github

In [ ]:
import pandas as pd
df_slneg = pd.read_csv("https://raw.githubusercontent.com/rafikingakbar/BrokerAppSentiment/refs/heads/main/data/reviews_data_SentimenNegative.csv")
df_slneg

###3.Stopwords and Cleaning

a. Mencari kata yang akan dimasukkan dalam kamus Stopwords

In [ ]:
from collections import Counter

all_words = " ".join(df_slneg['review_all']).split()
word_freq = Counter(all_words)

word_freq.most_common(150)


b. Stopwords

In [ ]:
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')

custom_stopwords = {
    "tidak","di","ini","sudah","saya","bisa","yang","nya",
    "dan","saja","ada","ke","lagi","kalau","dari","jadi",
    "mau","sangat","banget", "ok", "sip", "ya"
}

sw = set(stopwords.words('indonesian'))
sw = sw.union(custom_stopwords)

def remove_stopwords(text):
    tokens = str(text).split()
    tokens = [word for word in tokens if word not in sw]
    return " ".join(tokens)

df_slneg['review_all'] = df_slneg['review_all'].apply(remove_stopwords)
df_slneg

#**Feature Engginering**

###1.TF-IDF

a. Membentuk TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

documents = df_slneg['review_all'].astype(str).tolist()

vectorizer = TfidfVectorizer(
    token_pattern=r'(?u)\b[a-zA-Z]{2,}\b'
)
X = vectorizer.fit_transform(documents)

df_TFIDF = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out()
)

print("Shape TF-IDF:", X.shape)
df_TFIDF

#**Clustering**

###1.K-Means

a. Liat Inertia

In [ ]:
from sklearn.cluster import KMeans
import pandas as pd

k_range = range(2, 10)
inertia = []

for k in k_range:
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    km.fit(X)
    inertia.append(km.inertia_)

df_inertia = pd.DataFrame({
    "k": list(k_range),
    "inertia": inertia
})

df_inertia

b. Visualisasi Elbow

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(df_inertia['k'], df_inertia['inertia'], marker='o')
plt.title("Metode Elbow - K-Means")
plt.xlabel("Jumlah Cluster (k)")
plt.ylabel("Inertia (SSE)")
plt.grid(True)
plt.show()

c. Silhoutte Score

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []

for k in k_range:
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    labels = km.fit_predict(X)
    sil = silhouette_score(X, labels)
    silhouette_scores.append(sil)

df_silhouette = pd.DataFrame({
    "k": list(k_range),
    "silhouette": silhouette_scores
})

df_silhouette

d. DBI

In [ ]:
from sklearn.metrics import davies_bouldin_score

dbi_scores = []

for k in k_range:
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    labels = km.fit_predict(X)
    dbi = davies_bouldin_score(X.toarray(), labels)
    dbi_scores.append(dbi)

df_dbi = pd.DataFrame({
    "k": list(k_range),
    "dbi": dbi_scores
})

df_dbi

e. Gabungan Evaluasi Kuantitatif K-Means

In [ ]:
df_kmeans_eval = df_inertia.merge(df_silhouette, on="k").merge(df_dbi, on="k")
df_kmeans_eval

**Setelah mempertimbangkan hasil yang diperoleh, kami memutuskan menggunakan K-Means dengan K=6.**

f. K-Means FInal (k=6)

In [ ]:
from sklearn.cluster import KMeans

k_opt = 6

kmeans_final = KMeans(
    n_clusters=k_opt,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans_final.fit_predict(X)

df_slneg['cluster_kmeans'] = cluster_labels

df_slneg[['review_all', 'cluster_kmeans']]

g. Top words per cluster

In [ ]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()
centers = kmeans_final.cluster_centers_
top_n = 5

top_words_per_cluster = {}

for c in range(k_opt):
    idx = np.argsort(centers[c])[::-1][:top_n]
    top_words = [feature_names[i] for i in idx]
    top_words_per_cluster[c] = top_words

    print(f"\nCluster {c}")
    print(", ".join(top_words))

###2.Agglomerative Hierarchical Clustering

**Penentuan jumlah klaster (k=6) dilakukan berdasarkan hasil K-Means untuk menjaga kesetaraan dalam analisis (apple to apple).**

a. Menjalankan Agglomerative (k=6)

In [ ]:
from sklearn.cluster import AgglomerativeClustering

k_agg = 6
agglo = AgglomerativeClustering(
    n_clusters=k_agg,
    linkage='average',
    metric='euclidean'
)

cluster_agglo = agglo.fit_predict(X.toarray())

df_slneg['cluster_agglo'] = cluster_agglo

df_slneg[['review_all', 'cluster_kmeans', 'cluster_agglo']]

b. Hitung Silhoutte & DBI untuk Agglomerative

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

sil_agglo = silhouette_score(X, cluster_agglo)
dbi_agglo = davies_bouldin_score(X.toarray(), cluster_agglo)

sil_agglo, dbi_agglo

c. Top words per cluster

In [ ]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()
top_n = 5

for c in range(k_agg):
    idx_cluster = np.where(cluster_agglo == c)[0]

    mean_tfidf = X[idx_cluster].mean(axis=0).A1
    top_idx = mean_tfidf.argsort()[::-1][:top_n]

    top_words = [feature_names[i] for i in top_idx]

    print(f"\nCluster {c} (Agglo Average + Euclidean)")
    print(", ".join(top_words))

#**Visualisasi Scatter Plot Metode Clustering**

###1.PCA 2D dari TF-IDF

In [ ]:
from sklearn.decomposition import PCA
import pandas as pd

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X.toarray())

df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca['cluster_kmeans'] = df_slneg['cluster_kmeans'].values
df_pca['cluster_agglo'] = df_slneg['cluster_agglo'].values


###2.Scatter K-Means vs Agglomerative

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# K-Means
plt.subplot(1, 2, 1)
plt.scatter(
    df_pca['PC1'], df_pca['PC2'],
    c=df_pca['cluster_kmeans'], s=8
)
plt.title('PCA Scatter Plot - K-Means')
plt.xlabel('PC1'); plt.ylabel('PC2')

# Agglomerative
plt.subplot(1, 2, 2)
plt.scatter(
    df_pca['PC1'], df_pca['PC2'],
    c=df_pca['cluster_agglo'], s=8
)
plt.title('PCA Scatter Plot - Agglomerative')
plt.xlabel('PC1'); plt.ylabel('PC2')

plt.tight_layout()
plt.show()


#**Analisis Ulasan Negatif**

**Hasil yang didapatkan menunjukkan bahwa untuk analisis ulasan negatif, kami menggunakan metode clustering K-Means.**

In [ ]:
import pandas as pd

#1.Tabel jumlah
tab_count = pd.crosstab(df_slneg['cluster_kmeans'],
                        df_slneg['app_name'])
print("Jumlah review per aplikasi di tiap cluster:")
print(tab_count)

#2. Tabel proporsi
tab_prop = tab_count.div(tab_count.sum(axis=1), axis=0)
print("\nProporsi aplikasi di tiap cluster (per baris, total = 1):")
print(tab_prop)

#3. Tabel persen
tab_prop_percent = tab_prop * 100
print("\nProporsi aplikasi di tiap cluster (dalam %):")
print(tab_prop_percent.round(2))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

#Tabel jumlah
tab_count = pd.crosstab(df_slneg['cluster_kmeans'],
                        df_slneg['app_name'])

#Ubah ke proporsi per cluster
tab_prop = tab_count.div(tab_count.sum(axis=1), axis=0)

#Plot stacked bar
tab_prop.plot(kind='bar', stacked=True, figsize=(10,6))

plt.title('Proporsi Aplikasi pada Setiap Cluster (Ulasan Negatif)')
plt.xlabel('Cluster')
plt.ylabel('Proporsi')
plt.legend(title='Aplikasi')
plt.tight_layout()
plt.show()